<a href="https://colab.research.google.com/github/LSC18/uds-nrc-finetuning-dataset/blob/main/notebooks/qlora_v3_colab_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UDS NRC v3 QLoRA Smoke Test

순서대로 실행한다. 전체 학습 전에 1.5B base baseline과 50-step smoke만 확인한다.

In [7]:
!nvidia-smi
%cd /content
!test -d uds-nrc-finetuning-dataset/.git && git -C uds-nrc-finetuning-dataset pull --ff-only || git clone https://github.com/LSC18/uds-nrc-finetuning-dataset.git
%cd uds-nrc-finetuning-dataset

Wed Sep 23 06:58:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
# Colab의 CUDA 호환 PyTorch 버전을 자동 고정한 뒤 나머지 패키지만 설치한다.
!python scripts/bootstrap_colab.py

+ /usr/bin/python3 -m pip uninstall -y -q torchvision torchaudio torchtext
+ /usr/bin/python3 -m pip install -q --constraint /tmp/uds-colab-79hxwjat/constraints.txt --requirement /content/uds-nrc-finetuning-dataset/requirements-colab.txt
{
  "status": "ready",
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "cuda_available": true,
  "gpu": "Tesla T4",
  "accelerate": "1.15.0",
  "bitsandbytes": "0.50.2",
  "datasets": "5.0.1",
  "peft": "0.21.0",
  "transformers": "5.8.1",
  "trl": "1.13.0"
}


In [9]:
!python scripts/validate_dataset.py --data-dir full_v3
!shasum -a 256 -c SHA256SUMS
!python scripts/preflight.py --config configs/qlora_v3_smoke_1.5b.json --report reports/preflight_v3_1.5b_gpu.json

{"status": "OK", "data_dir": "full_v3", "split_counts": {"train": 8266, "validation": 777, "test": 1364}}
full_v2/episodes.jsonl: OK
full_v2/metadata.json: OK
full_v2/test.jsonl: OK
full_v2/train.jsonl: OK
full_v2/validation.jsonl: OK
full_v3/episodes.jsonl: OK
full_v3/metadata.json: OK
full_v3/test.jsonl: OK
full_v3/train.jsonl: OK
full_v3/validation.jsonl: OK
{
  "status": "ready",
  "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
  "data_dir": "/content/uds-nrc-finetuning-dataset/full_v3",
  "max_length": 512,
  "truncation_count": 0,
  "token_stats": {
    "train": {
      "count": 8266,
      "min": 89,
      "median": 138,
      "p95": 196,
      "p99": 228,
      "max": 228,
      "over_max_length": 0
    },
    "validation": {
      "count": 777,
      "min": 91,
      "median": 135,
      "p95": 196,
      "p99": 196,
      "max": 228,
      "over_max_length": 0
    },
    "test": {
      "count": 1364,
      "min": 90,
      "median": 138,
      "p95": 202,
      "p99": 202,
    

## Base model baseline
먼저 held-out test 20개로 generation 경로를 확인한다.

In [10]:
!python evaluate_exact_match.py --model-name Qwen/Qwen2.5-1.5B-Instruct --data-dir full_v3 --limit 20 --output reports/v3_base_1.5b_first20.jsonl

Loading weights: 100% 338/338 [00:01<00:00, 286.54it/s]
{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "adapter": false,
  "data_dir": "full_v3",
  "samples": 20,
  "correct": 0,
  "exact_match": 0.0,
  "exact_match_by_scenario": {
    "nrc_invalid_key_recovery": 0.0,
    "nrc_length_recovery": 0.0,
    "nrc_range_recovery": 0.0,
    "nrc_sequence_recovery": 0.0,
    "write_probe_recovery": 0.0
  },
  "exact_match_by_ecu_profile": {
    "security_only_heldout": 0.0
  },
  "predictions": "reports/v3_base_1.5b_first20.jsonl"
}


## 50-step QLoRA smoke
adapter와 final_metrics.json이 생성되고 loss가 NaN이 아니면 성공이다.

In [11]:
!python train_qlora.py --config configs/qlora_v3_smoke_1.5b.json

Converting to completion-only training format: 100% 8266/8266 [00:00<00:00, 14013.99 examples/s]
Converting to completion-only training format: 100% 777/777 [00:00<00:00, 14265.72 examples/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Loading weights: 100% 338/338 [00:04<00:00, 69.21it/s]
Tokenizing train dataset: 100% 8266/8266 [00:09<00:00, 876.83 examples/s]
Building labels for train dataset: 100% 8266/8266 [00:02<00:00, 4003.06 examples/s]
Truncating train dataset: 100% 8266/8266 [00:01<00:00, 5026.70 examples/s]
Dropping fully masked examples from train dataset: 100% 8266/8266 [00:01<00:00, 6882.10 examples/s]
Tokenizing eval dataset: 100% 777/777 [00:00<00:00, 984.65 examples/s] 
Building labels for eval dataset: 100% 777/777 [00:00<00:00, 4235.59 examples/s]
Truncating eval dataset: 100% 777/777 [00:00<00:00, 5133.52 examples/s]
Dropping fully masked examples from eval dataset: 100% 777/777 [00:00<00:00, 7380.07 examples/s]

In [12]:
!python evaluate_exact_match.py --adapter-path outputs/v3-smoke-1.5b --data-dir full_v3 --output reports/v3_smoke_1.5b.jsonl
!cat outputs/v3-smoke-1.5b/final_metrics.json

Loading weights: 100% 338/338 [00:01<00:00, 248.17it/s]
{
  "model": "outputs/v3-smoke-1.5b",
  "adapter": true,
  "data_dir": "full_v3",
  "samples": 1364,
  "correct": 878,
  "exact_match": 0.6436950146627566,
  "exact_match_by_scenario": {
    "failure": 0.4166666666666667,
    "nrc_invalid_key_recovery": 0.4984025559105431,
    "nrc_length_recovery": 0.4489795918367347,
    "nrc_range_recovery": 0.9206896551724137,
    "nrc_sequence_recovery": 0.0,
    "security_probe_recovery": 1.0,
    "write_probe_recovery": 0.6629422718808193
  },
  "exact_match_by_ecu_profile": {
    "security_only_heldout": 0.6436950146627566
  },
  "predictions": "reports/v3_smoke_1.5b.jsonl"
}
{
  "train_runtime": 966.9623,
  "train_samples_per_second": 0.414,
  "train_steps_per_second": 0.052,
  "total_flos": 445737132936192.0,
  "train_loss": 0.6779294753074646,
  "epoch": 0.04839099927413501,
  "final_eval_loss": 0.4944586157798767,
  "final_eval_runtime": 245.289,
  "final_eval_samples_per_second": 3.16

## 다음 단계
smoke가 성공한 경우에만 `configs/qlora_v3_full_1.5b.json`으로 전체 학습한다. 출력 폴더와 reports를 먼저 Google Drive에 복사한다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")